# 05. 멀티 인덱스 통합 검색 및 Cross-Index RAG

4개 법률 인덱스(판례·헌재결정례·법제처해석례·행정심판재결례)에 대해 다양한 검색 기법을 실습합니다.

## 실습 내용
1. **필터 검색** - 메타데이터 기반 필터링 (심급, 결정유형, 관련법령 등)
2. **하이브리드 검색** - 키워드 + 벡터 결합
3. **Cross-Index 검색** - 4개 인덱스 동시 검색
4. **Cross-Index RAG** - 여러 법률 소스를 종합한 GPT-5.4 응답 생성

## 1. 환경 설정

In [ ]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

from src.search.legal_indexes import LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX
from src.preprocessing.embedding import EmbeddingGenerator

mgr = LegalIndexManager(
    endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'],
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

embedder = EmbeddingGenerator(
    endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    deployment=os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large'),
)

# 인덱스 통계
for s in mgr.get_all_stats():
    print(f"  {s['index_name']}: {s['document_count']}건")

## 2. 필터 검색 데모

메타데이터 필터를 활용한 정밀 검색입니다. OData 필터 문법을 사용합니다.

In [ ]:
# 2-1. 판례: 대법원(3심) 세무 사건만 필터
print("── 판례: 대법원 세무 사건 ──")
results = mgr.search(
    PREC_INDEX,
    query="*",
    filter_expr="courtLevel eq '3심' and caseType eq '세무'",
    select=["caseName", "caseNumber", "judgmentDate", "courtName"],
    top=10,
)
for r in results:
    print(f"  [{r['caseNumber']}] {r['caseName'][:60]}")

print(f"\n총 {len(results)}건")

# 2-2. 헌재결정례: 위헌 결정만 필터
print("\n── 헌재: 위헌 결정 ──")
results = mgr.search(
    CONST_INDEX,
    query="*",
    filter_expr="decisionType eq '위헌'",
    select=["caseName", "caseNumber", "decisionType"],
    top=5,
)
for r in results:
    print(f"  [{r['caseNumber']}] {r['caseName'][:60]} → {r['decisionType']}")

# 2-3. 행정심판: 인용(승소) 재결만 필터
print("\n── 행정심판: 인용 재결 ──")
results = mgr.search(
    ADMIN_INDEX,
    query="*",
    filter_expr="decisionType eq '인용'",
    select=["caseName", "caseNumber", "committee", "order"],
    top=5,
)
for r in results:
    print(f"  [{r['committee']}] {r['caseName'][:60]}")
    print(f"    주문: {r['order'][:80]}")

## 3. Cross-Index 통합 검색

4개 인덱스를 동시에 검색하여 관련성 높은 결과를 통합 정렬합니다.

In [ ]:
INDEX_LABELS = {
    PREC_INDEX: "판례",
    CONST_INDEX: "헌재결정례",
    INTERP_INDEX: "법제처해석례",
    ADMIN_INDEX: "행정심판재결례",
}

def cross_index_search(query: str, top_per_index: int = 3) -> list[dict]:
    """4개 인덱스 통합 검색 (score 기반 정렬)"""
    embedding = embedder.generate(query)
    all_results = []
    for idx_name in [PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX]:
        results = mgr.search(idx_name, query=query, embedding=embedding, top=top_per_index)
        for r in results:
            r["_index"] = idx_name
            r["_label"] = INDEX_LABELS[idx_name]
        all_results.extend(results)
    # score 높은 순 정렬
    all_results.sort(key=lambda x: x.get("score", 0), reverse=True)
    return all_results

# 테스트: "개인정보 보호" 관련 4개 인덱스 통합 검색
query = "개인정보 보호와 관련된 법률적 쟁점"
results = cross_index_search(query, top_per_index=3)

print(f"검색어: '{query}'")
print(f"총 {len(results)}건 (4개 인덱스 통합)\n")

for i, r in enumerate(results[:8], 1):
    title = r.get("caseName") or r.get("title", "N/A")
    summary_field = r.get("summary") or r.get("reply") or r.get("reasonSummary") or ""
    print(f"{i}. [{r['_label']}] (score: {r['score']:.4f})")
    print(f"   {title[:80]}")
    print(f"   {summary_field[:120]}...")
    print()

## 4. Cross-Index RAG

여러 법률 소스(판례, 헌재결정례, 해석례, 재결례)를 종합하여 GPT-5.4가 통합 답변을 생성합니다.

In [ ]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

openai_client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    azure_ad_token_provider=token_provider,
    api_version='2024-10-21',
)

def _format_result(r: dict) -> str:
    """검색 결과를 컨텍스트 문자열로 변환"""
    label = r.get("_label", "기타")
    title = r.get("caseName") or r.get("title", "")
    case_no = r.get("caseNumber") or r.get("docNumber", "")

    body_parts = []
    for field in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]:
        if r.get(field):
            body_parts.append(r[field])
    body = "\n".join(body_parts)

    return f"[{label}] {title} ({case_no})\n{body}"


def cross_index_rag(question: str, top_per_index: int = 3) -> str:
    """Cross-Index RAG: 4개 인덱스 검색 → GPT-5.4 응답"""
    results = cross_index_search(question, top_per_index=top_per_index)

    context = "\n\n---\n\n".join([_format_result(r) for r in results[:8]])

    response = openai_client.chat.completions.create(
        model=os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4'),
        messages=[
            {
                'role': 'system',
                'content': (
                    '당신은 한국 법률 전문가입니다. '
                    '판례, 헌법재판소 결정례, 법제처 해석례, 행정심판 재결례 등 '
                    '다양한 법률 자료를 종합하여 사용자의 질문에 정확하고 체계적으로 답변하세요.\n\n'
                    '답변 시 반드시:\n'
                    '1. 근거가 되는 판례/결정례/해석례의 사건번호를 명시하세요\n'
                    '2. 법률 자료의 종류(판례/헌재결정례/해석례/재결례)를 구분하여 설명하세요\n'
                    '3. 검색 결과에 없는 내용은 답변하지 마세요'
                ),
            },
            {
                'role': 'user',
                'content': f'## 검색된 법률 자료\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_tokens=2000,
        temperature=0.3,
    )

    return response.choices[0].message.content

print("Cross-Index RAG 함수 준비 완료")

In [ ]:
from IPython.display import Markdown, display

# RAG 테스트 1: 개인정보 관련 종합 질의
q1 = "개인정보 보호와 관련하여 법원 판례, 헌법재판소 결정, 법제처 해석례에서 각각 어떤 입장을 취하고 있나요?"
answer1 = cross_index_rag(q1)
display(Markdown(f"### 질문 1\n{q1}\n\n### 답변\n{answer1}"))

In [ ]:
# RAG 테스트 2: 세무/조세 관련 종합 질의
q2 = "조세법 해석에서 실질과세원칙이 적용된 사례와 그 기준을 설명해주세요."
answer2 = cross_index_rag(q2)
display(Markdown(f"### 질문 2\n{q2}\n\n### 답변\n{answer2}"))

In [ ]:
# RAG 테스트 3: 행정처분 관련
q3 = "행정처분에 불복하는 경우 행정심판에서 인용(승소)되려면 어떤 요건이 필요한가요?"
answer3 = cross_index_rag(q3)
display(Markdown(f"### 질문 3\n{q3}\n\n### 답변\n{answer3}"))

## 5. 인덱스 통계 요약

In [ ]:
print("=" * 50)
print("  법률 멀티 인덱스 현황")
print("=" * 50)

total = 0
for s in mgr.get_all_stats():
    label = INDEX_LABELS.get(s['index_name'], s['index_name'])
    emb_fields = mgr.EMBEDDING_FIELDS.get(s['index_name'], [])
    count = s['document_count']
    total += count
    print(f"  {label:12s} ({s['index_name']}): {count:3d}건  임베딩={emb_fields}")

print(f"\n  총 문서: {total}건")
print(f"  벡터 차원: 3072 (text-embedding-3-large)")
print(f"  비용 최적화: 요지/핵심 필드만 임베딩")
print("=" * 50)

---
## Lab 완료!

이 Hands-on Lab을 통해 학습한 내용:
1. **비용 최적화 스키마 설계**: 전체 문서가 아닌 핵심 필드만 벡터화
2. **다중 인덱스 관리**: 4개 법률 데이터 소스별 최적화된 인덱스
3. **메타데이터 필터 검색**: OData 필터를 활용한 정밀 검색
4. **Cross-Index 통합 검색**: 여러 인덱스 결과의 score 기반 통합
5. **Cross-Index RAG**: 다양한 법률 소스를 종합한 AI 응답 생성